# Direct Prediction — Recovery of Missing Metrics (Chapter 4.1)

**Thesis section.** 4.1 — fills in the model/ticker cells where the first pass crashed

**Inputs.** `results/ch2_standardised_results.json` (partial)

**Outputs.** `results/ch2_standardised_results.json` (complete)

**Expected runtime.** ~20 min. **Expected GPU.** T4.

> All paths in the CONFIG cell below resolve relative to the repo root. On
> Google Colab, uncomment the Drive fallback line.


In [ ]:
# === CONFIG (edit paths here) ===
from pathlib import Path

CONFIG = {
    "data_dir":        Path("../../data"),         # processed + features + splits
    "results_dir":     Path("../../results"),
    "checkpoints_dir": Path("../../checkpoints"),
    "seed":            42,
    "device":          "cuda",                      # or "cpu"
    # Colab fallback — uncomment if running on Colab with the dataset mounted:
    # "data_dir": Path("/content/drive/MyDrive/thesis_data"),
}
for key, path in CONFIG.items():
    if isinstance(path, Path):
        path.mkdir(parents=True, exist_ok=True)


# Chapter 2 — Missing Metrics Only## COMP3931 · Heliang Li (sc21hl)**Purpose:** Fill in ONLY the missing metrics from Table 2.1.This is a targeted notebook — it does NOT re-run models that already have complete results.### Missing Metrics:| Model | Missing ||-------|---------|| VMD-LSTM (Global) | N, RMSE || VMD-LSTM (Rolling) | N, RMSE, R² || Wavelet-LSTM | N, RMSE, R² || LightGBM v2 (Alpha158) | RMSE, R² |All other models have complete metrics from prior runs.

In [ ]:
# ── Setup ──
import warnings, os, json, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

from google.colab import drive
drive.mount('/content/drive')

SPLITS_DIR = CONFIG['data_dir'] / r'splits'
FREQ = '1hour'
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'TSLA', 'SPY', 'QQQ']
SEQ_LEN = 30
BATCH_SIZE = 64
LR = 1e-3
EPOCHS = 100
PATIENCE = 10

missing_metrics = {}

In [ ]:
# ── Shared Utilities (copy from main notebook) ──

def load_splits(ticker, freq=FREQ):
    base = os.path.join(SPLITS_DIR, f"{ticker}_{freq}")
    train = pd.read_csv(os.path.join(base, 'train.csv'), parse_dates=['ts_event'])
    val   = pd.read_csv(os.path.join(base, 'val.csv'),   parse_dates=['ts_event'])
    test  = pd.read_csv(os.path.join(base, 'test.csv'),  parse_dates=['ts_event'])
    return train, val, test

def load_close_series(ticker, freq=FREQ):
    train, val, test = load_splits(ticker, freq)
    return train['close'], val['close'], test['close']

def prepare_sequences(X, y, seq_len=SEQ_LEN):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return np.array(Xs), np.array(ys)

def da_z_test(da, n, null=0.5):
    se = np.sqrt(null * (1 - null) / n)
    z = (da - null) / se
    p = 1 - stats.norm.cdf(z)
    return z, p

def directional_accuracy(y_true, y_pred):
    true_dir = np.sign(np.diff(y_true))
    pred_dir = np.sign(np.diff(y_pred))
    mask = (true_dir != 0) & (pred_dir != 0)
    if mask.sum() == 0: return 0.5
    return (true_dir[mask] == pred_dir[mask]).mean()

def train_pytorch_model(model, train_X, train_y, val_X, val_y,
                        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, patience=PATIENCE):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    tX = torch.FloatTensor(train_X).to(device)
    ty = torch.FloatTensor(train_y).unsqueeze(-1).to(device)
    vX = torch.FloatTensor(val_X).to(device)
    vy = torch.FloatTensor(val_y).unsqueeze(-1).to(device)
    train_dl = DataLoader(TensorDataset(tX, ty), batch_size=batch_size, shuffle=True)
    best_loss, best_state, wait = float('inf'), None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(vX), vy).item()
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"    Early stop epoch {epoch+1}")
                break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    return model

print("Utilities loaded.")

---## LightGBM v2 (Alpha158) — Missing: RMSE, R²

In [ ]:
# ── LightGBM v2: Get RMSE and R² ──
import lightgbm as lgb

def compute_alpha158_features(df):
    o, h, l, c, v = df['open'], df['high'], df['low'], df['close'], df['volume']
    feat = pd.DataFrame(index=df.index)
    feat['close_open'] = c / o - 1
    feat['high_low'] = h / l - 1
    feat['close_high'] = c / h - 1
    feat['close_low'] = c / l - 1
    for w in [5, 10, 20, 30, 60]:
        feat[f'ma_{w}'] = c.rolling(w).mean() / c - 1
        feat[f'std_{w}'] = c.rolling(w).std() / c
        feat[f'ret_{w}'] = c.pct_change(w)
        feat[f'vol_ma_{w}'] = v.rolling(w).mean() / (v + 1e-8) - 1
    for w in [6, 12, 24]:
        delta = c.diff()
        gain = delta.clip(lower=0).rolling(w).mean()
        loss = (-delta.clip(upper=0)).rolling(w).mean()
        feat[f'rsi_{w}'] = gain / (gain + loss + 1e-8)
    ema12 = c.ewm(span=12).mean()
    ema26 = c.ewm(span=26).mean()
    feat['macd'] = (ema12 - ema26) / c
    feat['macd_signal'] = feat['macd'].ewm(span=9).mean()
    feat['macd_hist'] = feat['macd'] - feat['macd_signal']
    ma20 = c.rolling(20).mean(); std20 = c.rolling(20).std()
    feat['bb_upper'] = (ma20 + 2*std20) / c - 1
    feat['bb_lower'] = (ma20 - 2*std20) / c - 1
    feat['bb_width'] = (4*std20) / (ma20 + 1e-8)
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    feat['atr_14'] = tr.rolling(14).mean() / c
    return feat

LGB_PARAMS = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'num_leaves': 63, 'learning_rate': 0.05, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 5, 'verbose': -1,
    'n_estimators': 500, 'early_stopping_rounds': 30
}

print("LightGBM v2 (Alpha158) — Collecting RMSE and R²")
print("="*60)

all_train, all_val, all_test = [], [], []
ticker_ranges = {}

for ticker in TICKERS:
    train, val, test = load_splits(ticker)
    for df in [train, val, test]:
        af = compute_alpha158_features(df)
        for col in af.columns: df[col] = af[col].values
        df['target'] = df['close'].shift(-1)
    train, val, test = train.dropna(), val.dropna(), test.dropna()
    start = sum(len(t) for t in all_test)
    ticker_ranges[ticker] = (start, start + len(test))
    all_train.append(train); all_val.append(val); all_test.append(test)

pool_tr = pd.concat(all_train, ignore_index=True)
pool_va = pd.concat(all_val, ignore_index=True)
pool_te = pd.concat(all_test, ignore_index=True)

feat_cols = [c for c in pool_tr.columns
             if c not in ['ts_event','close','target','ticker','_ticker','open','high','low','volume']
             and pool_tr[c].dtype in ['float64','float32','int64']]

dtrain = lgb.Dataset(pool_tr[feat_cols], pool_tr['target'])
dval = lgb.Dataset(pool_va[feat_cols], pool_va['target'], reference=dtrain)
model = lgb.train(LGB_PARAMS, dtrain, num_boost_round=500,
                  valid_sets=[dval], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])

all_preds = model.predict(pool_te[feat_cols])

lgbv2_results = []
for ticker in TICKERS:
    s, e = ticker_ranges[ticker]
    y_true = pool_te.iloc[s:e]['target'].values
    y_pred = all_preds[s:e]
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    da = directional_accuracy(y_true, y_pred)
    n = len(y_true) - 1
    print(f"  {ticker}: RMSE={rmse:.4f}, R²={r2:.4f}, DA={da:.4f}, N={n}")
    lgbv2_results.append({'ticker': ticker, 'RMSE': rmse, 'R2': r2, 'DA': da, 'N': n})

missing_metrics['LightGBM v2 (Alpha158)'] = lgbv2_results
print(f"\nMean RMSE: {np.mean([r['RMSE'] for r in lgbv2_results]):.4f}")
print(f"Mean R²: {np.mean([r['R2'] for r in lgbv2_results]):.4f}")

---## VMD-LSTM (Global, LEAKAGE) — Missing: N, RMSE

In [ ]:
# ── VMD-LSTM Global: Get N and RMSE ──
from vmdpy import VMD

class SimpleVMDLSTM(nn.Module):
    def __init__(self, K=5, hidden=64, layers=1):
        super().__init__()
        self.lstms = nn.ModuleList([nn.LSTM(1, hidden, layers, batch_first=True) for _ in range(K)])
        self.fc = nn.Linear(hidden * K, 1)
    def forward(self, x):
        outs = []
        for k in range(x.shape[2]):
            o, _ = self.lstms[k](x[:, :, k:k+1])
            outs.append(o[:, -1, :])
        return self.fc(torch.cat(outs, dim=1))

print("VMD-LSTM (Global, LEAKAGE) — Collecting N and RMSE")
print("="*60)

vmd_global_results = []
for ticker in TICKERS:
    print(f"\n--- {ticker} ---")
    train_c, val_c, test_c = load_close_series(ticker)
    full = pd.concat([train_c, val_c, test_c]).values

    u, _, _ = VMD(full, 2000, 0, 5, 0, 1, 1e-7)
    K = u.shape[0]
    n_tr, n_va = len(train_c), len(val_c)

    tr_imfs = u[:, :n_tr].T
    va_imfs = u[:, n_tr:n_tr+n_va].T
    te_imfs = u[:, n_tr+n_va:].T

    trX_s, trY = prepare_sequences(tr_imfs, train_c.values)
    vaX_s, vaY = prepare_sequences(va_imfs, val_c.values)
    teX_s, teY = prepare_sequences(te_imfs, test_c.values)

    model = SimpleVMDLSTM(K=K)
    model = train_pytorch_model(model, trX_s, trY, vaX_s, vaY)

    with torch.no_grad():
        preds = model(torch.FloatTensor(teX_s).to(device)).cpu().numpy().flatten()

    rmse = np.sqrt(mean_squared_error(teY, preds))
    r2 = r2_score(teY, preds)
    da = directional_accuracy(teY, preds)
    n = len(teY) - 1
    print(f"  {ticker}: N={n}, RMSE={rmse:.4f}, R²={r2:.4f}, DA={da:.4f}")
    vmd_global_results.append({'ticker': ticker, 'N': n, 'RMSE': rmse, 'R2': r2, 'DA': da})

missing_metrics['VMD-LSTM (Global)'] = vmd_global_results
print(f"\nMean N: {int(np.mean([r['N'] for r in vmd_global_results]))}")
print(f"Mean RMSE: {np.mean([r['RMSE'] for r in vmd_global_results]):.4f}")

---## VMD-LSTM (Rolling) — Missing: N, RMSE, R²

In [ ]:
# ── VMD-LSTM Rolling: Get N, RMSE, R² ──
print("VMD-LSTM (Rolling) — Collecting N, RMSE, R²")
print("="*60)

vmd_rolling_results = []
for ticker in TICKERS:
    print(f"\n--- {ticker} ---")
    train_c, val_c, test_c = load_close_series(ticker)
    full = pd.concat([train_c, val_c, test_c]).values
    vmd_window = 120
    n_before_test = len(train_c) + len(val_c)

    # Rolling VMD for test
    test_imfs = []
    for t in range(len(test_c)):
        idx = n_before_test + t
        if idx < vmd_window: continue
        seg = full[idx-vmd_window:idx]
        try:
            u, _, _ = VMD(seg, 2000, 0, 5, 0, 1, 1e-7)
            test_imfs.append(u[:, -1])
        except:
            test_imfs.append(test_imfs[-1] if test_imfs else np.zeros(5))
    test_imfs = np.array(test_imfs)
    test_targets = test_c.values[:len(test_imfs)]

    # Train
    train_imfs = []
    for t in range(vmd_window, len(train_c)):
        seg = full[t-vmd_window:t]
        try:
            u, _, _ = VMD(seg, 2000, 0, 5, 0, 1, 1e-7)
            train_imfs.append(u[:, -1])
        except:
            train_imfs.append(train_imfs[-1] if train_imfs else np.zeros(5))
    train_imfs = np.array(train_imfs)
    train_targets = train_c.values[vmd_window:vmd_window+len(train_imfs)]

    # Val
    val_imfs = []
    for t in range(len(val_c)):
        idx = len(train_c) + t
        if idx < vmd_window: continue
        seg = full[idx-vmd_window:idx]
        try:
            u, _, _ = VMD(seg, 2000, 0, 5, 0, 1, 1e-7)
            val_imfs.append(u[:, -1])
        except:
            val_imfs.append(val_imfs[-1] if val_imfs else np.zeros(5))
    val_imfs = np.array(val_imfs)
    val_targets = val_c.values[:len(val_imfs)]

    if len(train_imfs) < SEQ_LEN + 10:
        print(f"  Skipping {ticker}: insufficient data"); continue

    trX_s, trY = prepare_sequences(train_imfs, train_targets[:len(train_imfs)])
    vaX_s, vaY = prepare_sequences(val_imfs, val_targets[:len(val_imfs)])
    teX_s, teY = prepare_sequences(test_imfs, test_targets[:len(test_imfs)])

    model = SimpleVMDLSTM(K=5)
    model = train_pytorch_model(model, trX_s, trY, vaX_s, vaY)

    with torch.no_grad():
        preds = model(torch.FloatTensor(teX_s).to(device)).cpu().numpy().flatten()

    rmse = np.sqrt(mean_squared_error(teY, preds))
    r2 = r2_score(teY, preds)
    da = directional_accuracy(teY, preds)
    n = len(teY) - 1
    print(f"  {ticker}: N={n}, RMSE={rmse:.4f}, R²={r2:.4f}, DA={da:.4f}")
    vmd_rolling_results.append({'ticker': ticker, 'N': n, 'RMSE': rmse, 'R2': r2, 'DA': da})

missing_metrics['VMD-LSTM (Rolling)'] = vmd_rolling_results

---## Wavelet-LSTM — Missing: N, RMSE, R²

In [ ]:
# ── Wavelet-LSTM: Get N, RMSE, R² ──
import pywt

def sliding_window_wavelet_denoise(series, window=120, wavelet='db4', level=3):
    result = np.full(len(series), np.nan)
    for t in range(window, len(series)):
        segment = series[t-window:t]
        coeffs = pywt.wavedec(segment, wavelet, level=level)
        sigma = np.median(np.abs(coeffs[-1])) / 0.6745
        threshold = sigma * np.sqrt(2 * np.log(window))
        coeffs_t = [coeffs[0]] + [pywt.threshold(c, threshold, mode='soft') for c in coeffs[1:]]
        denoised = pywt.waverec(coeffs_t, wavelet)
        result[t] = denoised[-1]
    return result

class WaveletLSTM(nn.Module):
    def __init__(self, input_dim=2, hidden=64, layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Linear(hidden//2, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

print("Wavelet-LSTM — Collecting N, RMSE, R²")
print("="*60)

wavelet_results = []
for ticker in TICKERS:
    print(f"\n--- {ticker} ---")
    train_c, val_c, test_c = load_close_series(ticker)
    full = pd.concat([train_c, val_c, test_c]).values
    denoised = sliding_window_wavelet_denoise(full)
    features = np.column_stack([full, np.nan_to_num(denoised)])

    n_tr, n_va = len(train_c), len(val_c)
    trX_s, trY = prepare_sequences(features[:n_tr], train_c.values)
    vaX_s, vaY = prepare_sequences(features[n_tr:n_tr+n_va], val_c.values)
    teX_s, teY = prepare_sequences(features[n_tr+n_va:], test_c.values)

    if len(trX_s) < 10:
        print(f"  Skipping {ticker}: insufficient data"); continue

    model = WaveletLSTM(input_dim=2)
    model = train_pytorch_model(model, trX_s, trY, vaX_s, vaY)

    with torch.no_grad():
        preds = model(torch.FloatTensor(teX_s).to(device)).cpu().numpy().flatten()

    rmse = np.sqrt(mean_squared_error(teY, preds))
    r2 = r2_score(teY, preds)
    da = directional_accuracy(teY, preds)
    n = len(teY) - 1
    print(f"  {ticker}: N={n}, RMSE={rmse:.4f}, R²={r2:.4f}, DA={da:.4f}")
    wavelet_results.append({'ticker': ticker, 'N': n, 'RMSE': rmse, 'R2': r2, 'DA': da})

missing_metrics['Wavelet-LSTM'] = wavelet_results

---## Summary of Missing Metrics

In [ ]:
# ── Summary ──
print("\n" + "="*60)
print("MISSING METRICS — FILLED")
print("="*60)

for model_name, rows in missing_metrics.items():
    print(f"\n{model_name}:")
    for r in rows:
        parts = [f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in r.items() if k != 'ticker']
        print(f"  {r['ticker']}: {', '.join(parts)}")
    ns = [r.get('N', 0) for r in rows]
    rmses = [r['RMSE'] for r in rows if 'RMSE' in r]
    r2s = [r['R2'] for r in rows if 'R2' in r]
    print(f"  MEAN — N={int(np.mean(ns))}, RMSE={np.mean(rmses):.4f}" +
          (f", R²={np.mean(r2s):.4f}" if r2s else ""))

# Save
with open('ch2_missing_metrics.json', 'w') as f:
    json.dump(missing_metrics, f, indent=2, default=str)
print("\nSaved to ch2_missing_metrics.json")